# Lot D — S4 : recherche dense multilingue (EXP-D-003)

**Auteur : Membre 5.** Compare BM25, la recherche dense et l'hybride (RRF) sur le jeu de dev
`tests/fixtures/rag_llm/`, pour plusieurs modèles d'embeddings.

1. *Exécution → Modifier le type d'exécution → GPU T4* (conseillé pour `bge-m3`, facultatif sinon).
2. Exécuter les cellules dans l'ordre.
3. Recopier les tableaux « Comparaison (GLOBAL) » et les seuils dans `experiments.md` (EXP-D-003).

## 1. Récupérer le projet
Le dépôt est privé : créer un token GitHub *classic* (case `repo`, 7 jours), le coller dans la case quand la cellule le demande, puis le supprimer sur GitHub après usage. **Ne jamais écrire le token dans une cellule.**

In [ ]:
# Dépôt privé : le token est demandé dans une case masquée, il n'est enregistré nulle part.
# (Alternative sans token : git archive -o NewsCheckTN.zip --prefix=NewsCheckTN/ HEAD,
#  puis files.upload() et !unzip -q NewsCheckTN.zip)
import re, subprocess
from getpass import getpass

token = getpass("Colle ton token GitHub puis appuie sur Entrée : ").strip()
url = f"https://{token}@github.com/SouissiOumaima/NewsCheckTN.git"
r = subprocess.run(["git", "clone", "-q", "-b", "lot-d-rag", url], capture_output=True, text=True)
del token, url
print("✅ Projet téléchargé" if r.returncode == 0
      else "❌ " + re.sub(r"gh[pousr]_\w+", "ghp_***", r.stderr))
%cd NewsCheckTN
!git remote set-url origin https://github.com/SouissiOumaima/NewsCheckTN.git


In [ ]:
!pip install -q -r requirements-dev.txt sentence-transformers

## 2. BM25 vs dense vs hybride — modèle par défaut (MiniLM multilingue)

In [ ]:
!python -m rag_llm.evaluate_retrieval --method all

## 3. Autres modèles d'embeddings
`e5` ajoute automatiquement les préfixes `query:` / `passage:`. `bge-m3` (≈ 2 Go) demande le GPU.

In [ ]:
for model in ["intfloat/multilingual-e5-small",
              "intfloat/multilingual-e5-base",
              "BAAI/bge-m3"]:
    print("\n" + "#" * 80 + f"\n# {model}")
    !python -m rag_llm.evaluate_retrieval --method all --model {model}

## 4. Seuil d'abstention en recherche dense
Le seuil `MIN_DENSE_SIMILARITY` est testé en mode **dense** (en hybride, le seuil ne sert qu'à filtrer avant la fusion).
Avec e5, les scores sont tassés entre 0,7 et 0,95 : une affirmation **sans** preuve peut avoir un score aussi haut qu'une bonne preuve (EXP-D-003). On regarde ici à partir de quel seuil la couverture s'effondre.

In [ ]:
MODEL = "intfloat/multilingual-e5-base"   # modèle retenu à l'étape 3
for seuil in ["0.3", "0.5", "0.7", "0.8"]:
    print(f"\n===== {MODEL} — MIN_DENSE_SIMILARITY = {seuil}")
    !MIN_DENSE_SIMILARITY={seuil} python -m rag_llm.evaluate_verdict --method dense --model {MODEL} | tail -12


## 5. Verdict de bout en bout avec le meilleur retrieval
Remplacer le modèle par celui qui a le meilleur Recall@5 à l'étape 3.

In [ ]:
!python -m rag_llm.evaluate_verdict --method bm25 | tail -12
!python -m rag_llm.evaluate_verdict --method hybrid --model intfloat/multilingual-e5-base | tail -12


## 6. Résultats
Les fichiers JSON sont dans `evaluation/results/` (non versionnés) : les télécharger depuis le panneau Fichiers si besoin.

In [ ]:
!ls evaluation/results/

## 7. Données réelles : Tunifact v1 (EXP-D-006)
Corpus construit à partir de la branche `lot-a1-data` (Lot A1, commit 77a00b9) : 6 articles Tunifact + 30 phrases TArC. Il n'est pas versionné : on le reconstruit ici. (`--version v0` pour refaire EXP-D-004.)

In [ ]:
!git fetch -q origin lot-a1-data:refs/remotes/origin/lot-a1-data
!python -m rag_llm.build_tunifact_eval

In [ ]:
CORPUS = "data/interim/lot_d/evidence_tunifact_v1.jsonl"
CLAIMS = "tests/fixtures/rag_llm/claims_tunifact_v1.jsonl"
for model in ["intfloat/multilingual-e5-base", "BAAI/bge-m3"]:
    print("\n" + "#" * 80 + f"\n# {model}")
    !python -m rag_llm.evaluate_retrieval --method all --model {model} --corpus {CORPUS} --claims {CLAIMS} --split tunifact-v1 --exp-id EXP-D-006

In [ ]:
!python -m rag_llm.evaluate_verdict --method hybrid --model intfloat/multilingual-e5-base --corpus {CORPUS} --claims {CLAIMS} --split tunifact-v1 --exp-id EXP-D-006 | tail -14